# Neural Machine Translation Demo with Marian
## Transformer Architecture

**By Ye Kyaw Thu, Lab Leader, Language Understanding Lab., Myanmar**  
**Date:** 26 May 2026  
*For AI (Fundamental) Class students*  

အရင်ဆုံး GPU အားမအား စစ်ပါ။  
(ဆရာ သုံးခဲ့တဲ့ GPU information ကို သိစေချင်လို့ run ပြတာလည်း ပါပါတယ်)

In [1]:
%pwd

'/d/Job_Requirements/PracticalProjectsFromDrYe/AIE-F/assignment-submission/class-22/TheinGiKyaw_Assignment7'

In [2]:
!nvidia-smi

Sun Jul 26 01:48:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 596.49         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   59C    P0             26W /   95W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Shell Script Preparation for Tranformer Architecture

Vocab ဖိုင်က ရှိပြီးသား ဖြစ်ရပါမယ်။  

In [3]:
from IPython.display import Markdown

# Read the file content
with open('./transformer.phmy1.sh', 'r') as f:
    content = f.read()

# Display it as a highlighted Bash block
display(Markdown(f"```bash\n{content}\n```"))

```bash
#!/bin/bash

## Based on the script displayed in Transformer-NMT-marian-ph2gp.ipynb.
## Local adaptation:
## - uses ./g2p-par instead of /home/ye/exp/nmt/marian-demo/g2p-par
## - uses GPU device(s); Marian must be built with CUDA support

set -euo pipefail

script_dir="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
model_folder="${script_dir}/model.transformer.phmy"
data_path="${DATA_PATH:-${script_dir}/g2p-par}"
src="ph"
tgt="my"
devices="${MARIAN_DEVICES:-0}"

mkdir -p "${model_folder}"

marian \
  --model "${model_folder}/model.npz" --type transformer \
  --train-sets "${data_path}/train.${src}" "${data_path}/train.${tgt}" \
  --max-length 200 \
  --vocabs "${data_path}/vocab/vocab.${src}.yml" "${data_path}/vocab/vocab.${tgt}.yml" \
  --mini-batch-fit -w 1000 --maxi-batch 100 \
  --early-stopping 10 \
  --valid-freq 5000 --save-freq 5000 --disp-freq 500 \
  --valid-metrics cross-entropy perplexity bleu \
  --valid-sets "${data_path}/dev.${src}" "${data_path}/dev.${tgt}" \
  --valid-translation-output "${model_folder}/valid.${src}-${tgt}.output" --quiet-translation \
  --valid-mini-batch 64 \
  --beam-size 6 --normalize 0.6 \
  --log "${model_folder}/train.log" --valid-log "${model_folder}/valid.log" \
  --enc-depth 2 --dec-depth 2 \
  --transformer-heads 8 \
  --transformer-postprocess-emb d \
  --transformer-postprocess dan \
  --transformer-dropout 0.3 --label-smoothing 0.1 \
  --learn-rate 0.0003 --lr-warmup 0 --lr-decay-inv-sqrt 16000 --lr-report \
  --clip-norm 5 \
  --tied-embeddings \
  --devices "${devices}" --seed 1111 \
  --exponential-smoothing \
  --dump-config > "${model_folder}/${src}-${tgt}.config.yml"

time marian -c "${model_folder}/${src}-${tgt}.config.yml" 2>&1 | tee "${model_folder}/transformer-${src}-${tgt}.log"

```

ဒီတခါတော့ architecture ကို transformer အဖြစ် ပြောင်းလိုက်ပါတယ်။  

```bash
--type transformer
```

## Training Transformer Model for Phoneme to Grapheme Translation

In [4]:
%pwd

'/d/Job_Requirements/PracticalProjectsFromDrYe/AIE-F/assignment-submission/class-22/TheinGiKyaw_Assignment7'

In [5]:
!./transformer.phmy1.sh

[2026-07-26 01:48:25] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-07-26 01:48:25] [marian] Running on LAPTOP-9DLBHLIH as process 5764 with command line:
[2026-07-26 01:48:25] [marian] marian -c /d/Job_Requirements/PracticalProjectsFromDrYe/AIE-F/assignment-submission/class-22/TheinGiKyaw_Assignment7/model.transformer.phmy/ph-my.config.yml
[2026-07-26 01:48:25] [config] after: 0e
[2026-07-26 01:48:25] [config] after-batches: 0
[2026-07-26 01:48:25] [config] after-epochs: 0
[2026-07-26 01:48:25] [config] all-caps-every: 0
[2026-07-26 01:48:25] [config] allow-unk: false
[2026-07-26 01:48:25] [config] authors: false
[2026-07-26 01:48:25] [config] beam-size: 6
[2026-07-26 01:48:25] [config] bert-class-symbol: "[CLS]"
[2026-07-26 01:48:25] [config] bert-mask-symbol: "[MASK]"
[2026-07-26 01:48:25] [config] bert-masking-fraction: 0.15
[2026-07-26 01:48:25] [config] bert-sep-symbol: "[SEP]"
[2026-07-26 01:48:25] [config] bert-train-type-embeddings: true
[2026-07-26 01:48:25

Transformer မော်ဒယ်ကတော့ ၁၁မိနစ်ပဲ ကြာပါတယ်။  
Epoch 874 မှာ training ပြီးသွားပါတယ်။  
Validation ဒေတာနဲ့ အကောင်းဆုံး ရလဒ်ကတော့ 76.63 BLEU score ပါ။   

## Checking Models  

In [6]:
!ls ./model.transformer.phmy

model.iter10000.npz  model.iter45000.npz      model.npz.progress.yml
model.iter15000.npz  model.iter50000.npz      model.npz.yml
model.iter20000.npz  model.iter5000.npz       ph-my.config.yml
model.iter25000.npz  model.iter55000.npz      train.log
model.iter30000.npz  model.npz		      transformer-ph-my.log
model.iter35000.npz  model.npz.decoder.yml    valid.log
model.iter40000.npz  model.npz.optimizer.npz  valid.ph-my.output


In [7]:
!cat ./model.transformer.phmy/valid.log

[2026-07-26 01:53:02] [valid] Ep. 80 : Up. 5000 : cross-entropy : 1.85265 : new best
[2026-07-26 01:53:03] [valid] Ep. 80 : Up. 5000 : perplexity : 1.61935 : new best
[2026-07-26 01:53:03] [valid] First sentence's tokens as scored:
[2026-07-26 01:53:03] [valid] DefaultVocab keeps original segments for scoring
[2026-07-26 01:53:03] [valid]   Hyp: ဥတ် တ ရ ဖ လ ဂု နီ
[2026-07-26 01:53:03] [valid]   Ref: ဥတ် တ ရ ဖ လ ဂု နီ
[2026-07-26 01:53:04] [valid] Ep. 80 : Up. 5000 : bleu : 75.4254 : new best
[2026-07-26 01:57:52] [valid] Ep. 159 : Up. 10000 : cross-entropy : 2.08689 : stalled 1 times (last best: 1.85265)
[2026-07-26 01:57:52] [valid] Ep. 159 : Up. 10000 : perplexity : 1.7211 : stalled 1 times (last best: 1.61935)
[2026-07-26 01:57:53] [valid] Ep. 159 : Up. 10000 : bleu : 75.7331 : new best
[2026-07-26 02:02:49] [valid] Ep. 239 : Up. 15000 : cross-entropy : 2.25777 : stalled 2 times (last best: 1.85265)
[2026-07-26 02:02:50] [valid] Ep. 239 : Up. 15000 : perplexity : 1.79935 : stalled 2

## Testing Transformer Model for Phoneme to Grapheme Translation

Marian ရဲ့ decoder command ကိုလည်း လေ့လာကြည့်ပါ။  

In [8]:
!marian-decoder --help

Marian: Fast Neural Machine Translation in C++
Usage: marian-decoder [OPTIONS]

General options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  --authors                             Print list of authors and exit
  --cite                                Print citation and exit
  --build-info TEXT                     Print CMake build options and exit. Set to 'all' to print advanced options
  -c,--config VECTOR ...                Configuration file(s). If multiple, later overrides earlier
  -w,--workspace INT=512                Preallocate arg MB of work space. Negative `--workspace -N` value allocates workspace as total available GPU memory minus N megabytes.
  --log TEXT                            Log training process information to file given by arg
  --log-level TEXT=info                 Set verbosity level of logging: trace, debug, info, warn, err(or), critical, off
  --log-time-zon

In [10]:
!time marian-decoder -m ./model.transformer.phmy/model.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0 < ./g2p-par/test.ph > ./transformer.phmy.hyp.txt

[2026-07-26 02:46:20] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-07-26 02:46:20] [marian] Running on LAPTOP-9DLBHLIH as process 6260 with command line:
[2026-07-26 02:46:20] [marian] marian-decoder -m ./model.transformer.phmy/model.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0
[2026-07-26 02:46:20] [config] alignment: ""
[2026-07-26 02:46:20] [config] allow-special: false
[2026-07-26 02:46:20] [config] allow-unk: false
[2026-07-26 02:46:20] [config] authors: false
[2026-07-26 02:46:20] [config] beam-size: 12
[2026-07-26 02:46:20] [config] bert-class-symbol: "[CLS]"
[2026-07-26 02:46:20] [config] bert-mask-symbol: "[MASK]"
[2026-07-26 02:46:20] [config] bert-masking-fraction: 0.15
[2026-07-26 02:46:20] [config] bert-sep-symbol: "[SEP]"
[2026-07-26 02:46:20] [config] bert-train-type-embeddings: true
[2026-07-26 02:46:20] [config] bert-type-vocab-size: 2
[2026-07-26 02:46:20] [config] best-deep: false
[2026-07-26 02:46:20] [config] buil

**Marian က training/testing အတွက် အရမ်းမြန်ပါတယ်။ အဲဒါကြောင့် ဆရာ ကြိုက်တယ်။**

## Evaluation

In [12]:
!perl /home/gi/tools/mosesbin/ubuntu-17.04/mosesdecoder/scripts/generic/multi-bleu.perl ./g2p-par/test.my < ./transformer.phmy.hyp.txt


BLEU = 76.69, 87.5/78.3/73.0/69.4 (BP=0.999, ratio=0.999, hyp_len=8041, ref_len=8048)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.


Sequence to Sequence နဲ့ Transformer မော်ဒယ်နှစ်ရဲ့ ရလဒ်ကို နှိုင်းယှဉ်ကြည့်တဲ့အခါမှာ Sequence to Sequence မော်ဒယ်ရဲ့ ရလဒ်က ပိုကောင်းတာကို တွေ့ရပါလိမ့်မယ်။  